In [1]:
import pandas as pd
import glob
from datetime import date

In [2]:
files = sorted(glob.glob("data/raw/nav_raw_*.txt"))
if not files:
    raise FileNotFoundError("No NAV files found in data/raw/. Run scraper.py first.")
raw_path = files[-1]
print(f"Using: {raw_path}")

Using: data/raw\nav_raw_20260822.txt


In [3]:
with open(raw_path, "r",encoding="utf-8")as f:
    raw_lines=f.readlines()
print(f"Total lines:{len(raw_lines)}")

Total lines:35734


In [4]:
for line in raw_lines[:30]:
    print(repr(line))

'Scheme Code;ISIN Div Payout/ ISIN Growth;ISIN Div Reinvestment;Scheme Name;Plan;Option;Net Asset Value;Date\n'
'\n'
' \n'
'\n'
'Open Ended Schemes(Debt Scheme - Banking and PSU Fund)\n'
'\n'
' \n'
'\n'
'Aditya Birla Sun Life Mutual Fund\n'
'\n'
' \n'
'\n'
'119551;INF209KA12Z1;INF209KA13Z9;Aditya Birla Sun Life Banking & PSU Debt Fund;Direct Plan;IDCW-Re-investment;106.8821;21-Aug-2026\n'
'\n'
'119552;INF209K01YM2;-;Aditya Birla Sun Life Banking & PSU Debt Fund;Direct Plan;MONTHLY DCW Payout;117.1807;21-Aug-2026\n'
'\n'
'119553;INF209K01YO8;-;Aditya Birla Sun Life Banking & PSU Debt Fund;Direct Plan;QUARTERLY IDCW Payout;104.9144;21-Aug-2026\n'
'\n'
'108272;INF209K01LX6;INF209KA11Z3;Aditya Birla Sun Life Banking & PSU Debt Fund;Regular Plan;IDCW-Re-investment;149.5533;21-Aug-2026\n'
'\n'
'110282;INF209K01LU2;-;Aditya Birla Sun Life Banking & PSU Debt Fund;Regular Plan;MONTHLY IDCW Payout;112.4288;21-Aug-2026\n'
'\n'
'108274;INF209K01LN7;-;Aditya Birla Sun Life Banking & PSU Debt Fund;R

In [5]:
#filter to real data rows
def is_data_row(line):
    parts=line.strip().split(";")
    return len(parts) >= 6 and parts[0].strip().isdigit()

def parse_row(line):
    parts = line.strip().split(";")
    scheme_code = parts[0].strip()
    isin_growth = parts[1].strip()
    is_reinvest = parts[2].strip()
    nav         = parts[-2].strip()   # always second-to-last
    nav_data    = parts[-1].strip()   # always last
    # everything between index 3 and -2 is the scheme name (handles 6 or 8+ column rows)
    scheme_name = " ".join(p.strip() for p in parts[3:-2] if p.strip())
    return [scheme_code, isin_growth, is_reinvest, scheme_name, nav, nav_data]

In [6]:
data_lines=[line for line in raw_lines if is_data_row(line)]
print(f"kept{len(data_lines)} out of {len(raw_lines)} lines")
print(f"dropped{len(raw_lines)-len(data_lines)} header/junk lines")

kept14282 out of 35734 lines
dropped21452 header/junk lines


In [7]:
cols=["scheme_code","isin_growth","is_reinvest","scheme_name","nav","nav_data"]
rows=[parse_row(line) for line in data_lines]
df=pd.DataFrame(rows,columns=cols)
df.head()


,scheme_code,isin_growth,is_reinvest,scheme_name,nav,nav_data
0,119551,INF209KA12Z1,INF209KA13Z9,Aditya Birla Sun Life Banking & PSU Debt Fund ...,106.8821,21-Aug-2026
1,119552,INF209K01YM2,-,Aditya Birla Sun Life Banking & PSU Debt Fund ...,117.1807,21-Aug-2026
2,119553,INF209K01YO8,-,Aditya Birla Sun Life Banking & PSU Debt Fund ...,104.9144,21-Aug-2026
3,108272,INF209K01LX6,INF209KA11Z3,Aditya Birla Sun Life Banking & PSU Debt Fund ...,149.5533,21-Aug-2026
4,110282,INF209K01LU2,-,Aditya Birla Sun Life Banking & PSU Debt Fund ...,112.4288,21-Aug-2026


In [8]:
df["nav"] = pd.to_numeric(df["nav"], errors="coerce")


In [9]:

# check how many rows broke during conversion, want to know before dropping them
bad_rows = df[df["nav"].isna()]
print(f"{len(bad_rows)} rows had a bad/missing nav value")
bad_rows.head()

0 rows had a bad/missing nav value


,scheme_code,isin_growth,is_reinvest,scheme_name,nav,nav_data


In [10]:
df["nav_date"] = pd.to_datetime(df["nav_data"], format="%d-%b-%Y", errors="coerce")

In [11]:
before = len(df)
df = df.dropna(subset=["nav", "nav_date"]).drop_duplicates()
print(f"{before} rows -> {len(df)} after dropping bad/dupe rows")


14282 rows -> 14282 after dropping bad/dupe rows


In [12]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 14282 entries, 0 to 14281
Data columns (total 7 columns):
 #   Column       Non-Null Count  Dtype         
---  ------       --------------  -----         
 0   scheme_code  14282 non-null  object        
 1   isin_growth  14282 non-null  object        
 2   is_reinvest  14282 non-null  object        
 3   scheme_name  14282 non-null  object        
 4   nav          14282 non-null  float64       
 5   nav_data     14282 non-null  object        
 6   nav_date     14282 non-null  datetime64[ns]
dtypes: datetime64[ns](1), float64(1), object(5)
memory usage: 781.2+ KB


In [13]:
import os
from datetime import date

today = date.today().strftime("%Y%m%d")
clean_path = f"data/clean/nav_clean_{today}.csv"
os.makedirs("data/clean", exist_ok=True)  # just in case the folder doesn't exist yet

In [14]:

df.to_csv(clean_path, index=False)
print(f"saved {len(df)} clean rows -> {clean_path}")

saved 14282 clean rows -> data/clean/nav_clean_20260825.csv
